# Render the full kiisafezone README showcase 🎬

Render the complete portrait composition from `assets/macos-readme-showcase.svg` using the Rust-based `svg-to-video` CLI. The source SVG is fetched from the latest `main` branch in this notebook.

**Target output:** 1080 × 1920 · 30 FPS · 59 seconds · H.264 MP4.

Run the cells from top to bottom. The renderer installation is needed only once per fresh Colab runtime.

In [ ]:
import os
import shutil
import subprocess
import urllib.request
from pathlib import Path

# Repair PATH first. A previous notebook cell may have replaced it with a literal '$PATH'.
system_paths = [
    "/root/.cargo/bin", "/usr/local/sbin", "/usr/local/bin",
    "/usr/sbin", "/usr/bin", "/sbin", "/bin",
]
existing_paths = os.environ.get("PATH", "").split(":")
fixed_paths = list(dict.fromkeys(p for p in system_paths + existing_paths if p and Path(p).is_dir()))
os.environ["PATH"] = ":".join(fixed_paths)
print("Repaired PATH:", os.environ["PATH"])

def run(command):
    print("\n$", " ".join(map(str, command)))
    subprocess.run(command, check=True)

# Stop with a clear message if this is not a Debian/Ubuntu-based Colab runtime.
apt_get = shutil.which("apt-get")
if apt_get is None:
    raise RuntimeError(
        "apt-get is still unavailable after PATH repair. This notebook is not running "
        "on a standard Linux Colab runtime. In Colab, choose Runtime > Change runtime type "
        "and use the hosted Python runtime, then rerun this cell."
    )

# Install compiler, build dependencies, FFmpeg, and download tools.
run([apt_get, "update", "-qq"])
run([apt_get, "install", "-y", "-qq", "build-essential", "gcc", "g++", "make", "pkg-config", "libssl-dev", "ffmpeg", "curl", "ca-certificates"])

# Install official Google Chrome (not Ubuntu's Snap Chromium placeholder).
chrome_deb = "/tmp/google-chrome-stable.deb"
urllib.request.urlretrieve(
    "https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb",
    chrome_deb,
)
run([apt_get, "install", "-y", "-qq", chrome_deb])

# Install Rust using Python's downloader; avoid relying on a possibly broken shell PATH.
rustup_script = urllib.request.urlopen("https://sh.rustup.rs").read().decode("utf-8")
bash = shutil.which("bash")
if bash is None:
    raise RuntimeError("bash is unavailable; cannot run the Rust installer.")
subprocess.run([bash, "-s", "--", "-y"], input=rustup_script, text=True, check=True)
os.environ["PATH"] = "/root/.cargo/bin:" + os.environ["PATH"]

# Verify prerequisites before compiling.
for command in ["cc", "google-chrome-stable", "ffmpeg", "cargo"]:
    executable = shutil.which(command)
    if executable is None:
        raise RuntimeError(f"Required command not found after installation: {command}")
    print(f"{command}: {executable}")

run([shutil.which("google-chrome-stable"), "--version"])
run([shutil.which("ffmpeg"), "-version"])
run([shutil.which("cargo"), "install", "svg-to-video", "--locked", "--jobs", "2"])

renderer = shutil.which("svg-to-video")
if renderer is None:
    raise RuntimeError("svg-to-video installation finished but the executable is not on PATH.")
run([renderer, "--help"])
print("\nRenderer installation and dependency checks passed.")

In [ ]:
# Fetch the latest showcase source from the repository's main branch
%cd /content
!rm -rf /content/kiisafezone
!git clone --depth 1 --branch main https://github.com/kiisafezone/kiisafezone.git
%cd /content/kiisafezone
!git log -1 --oneline
!ls -lh assets/macos-readme-showcase.svg

In [ ]:
# Validate the actual source SVG before rendering
from pathlib import Path
import xml.etree.ElementTree as ET

svg_path = Path('/content/kiisafezone/assets/macos-readme-showcase.svg')
assert svg_path.is_file(), f'Missing source SVG: {svg_path}'
root = ET.parse(svg_path).getroot()
print('Source:', svg_path)
print('Canvas:', root.attrib.get('width'), '×', root.attrib.get('height'))
print('ViewBox:', root.attrib.get('viewBox'))
print('File size:', f'{svg_path.stat().st_size:,}', 'bytes')
assert root.attrib.get('width') == '1080', 'Unexpected SVG width'
assert root.attrib.get('height') == '1920', 'Unexpected SVG height'
assert root.attrib.get('viewBox') == '0 0 1080 1920', 'Unexpected SVG viewBox'
print('Preflight passed.')

In [ ]:
# Render the complete 59-second README showcase
%cd /content/kiisafezone
!mkdir -p out
!rm -f out/showcase.mp4
!svg-to-video video assets/macos-readme-showcase.svg out/showcase.mp4 --duration 59 --fps 30 --width 1080 --height 1920 --codec h264
!ls -lh out/showcase.mp4

In [ ]:
# Verify the rendered video's dimensions, frame rate, and duration
!ffprobe -v error -select_streams v:0 -show_entries stream=width,height,r_frame_rate -show_entries format=duration -of default=noprint_wrappers=1 out/showcase.mp4

In [ ]:
# Download the finished showcase MP4
from google.colab import files
files.download('/content/kiisafezone/out/showcase.mp4')